---
title: Produce a Chap-ready CSV from Open Climate Service
short_title: Prepare data for Chap
---

Open Climate Service ships a companion workflow, `aggregate_to_chap_csv`, that performs the same org-unit aggregation but exports a **CHAP CSV** (`time_period`, `location`, and one column per variable) — the shape the [DHIS2 Chap Modeling Platform](https://chap.dhis2.org/) expects.

Needs the `open-climate-service` client (see the [section intro](intro.md)) and a running instance.

In [ ]:
from open_climate_service import ClimateService

service = ClimateService("https://my-instance.example.org")

## 1) Organisation unit boundaries

As before, provide a GeoJSON FeatureCollection of your org units. Each feature's `id` becomes the `location` in the CSV. See the [Organisation units](../guides/org-units/intro.md) guide.

In [ ]:
import json
from pathlib import Path

org_units = json.loads(Path("org-units.geojson").read_text())
print(len(org_units["features"]), "organisation units")

## 2) Run the workflow and save the CSV

This workflow returns a file rather than JSON. Passing `path=` to `execute()` writes the bytes to disk and returns the resulting `Path`.

In [ ]:
csv_path = service.execute(
    {
        "agg": {
            "process_id": "aggregate_to_chap_csv",
            "arguments": {
                "dataset_id": "era5land_precipitation_monthly",   # a published dataset id
                "temporal_extent": ["2025-01-01", "2025-12-31"],
                "geometries": org_units,
                "method": "mean",
                "period_type": "month",
            },
            "result": True,
        }
    },
    path="climate-monthly.csv",
)
csv_path

## 3) Inspect the result

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
df.head()

The CSV has `time_period`, `location` (your org-unit ids), and one column per variable — exactly the shape Chap consumes.

To combine this climate data with a health outcome (e.g. dengue cases) and population into a single modelling table, continue with the [Prepare data for Chap](../guides/import-chap/harmonize-to-chap.ipynb) guide, using this file as one of the harmonized inputs.